# Earth's elevation has two peaks. So does Mars's. Same reason?

**EPS 88 · PyEarth.** Open your own copy on DataHub: [click here](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2FAI4EPS%2FEPS88_PyEarth&branch=main&urlpath=lab%2Ftree%2FEPS88_PyEarth%2Fdocs/notebooks%2F03_two_peaks.ipynb).

Ask how high every point on a planet's solid surface is, and you might expect the answers to
pile up around one typical height, the way people's heights do. Earth's do not. Earth's surface
has two levels, with a gap between them that almost nothing sits in. That is a strange thing for
a planet to be, and it wants an explanation.

Mars has two levels as well. Mars has no ocean, no plate tectonics and no mid-ocean ridge — so
whatever built its two levels cannot be the thing that built Earth's. Or can it?

Today both planets arrive as a grid of numbers: one elevation for every one-degree square of the
surface. You will measure where each planet's two levels sit, draw them on a map, and decide
whether one story explains both. Then you will meet the other container Python keeps data in —
the table with named columns — on a catalogue of earthquakes.

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by an empty cell. Fill them all in, then export the notebook as a PDF and upload that.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Kernel → Restart Kernel and Run All Cells** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## What you'll be able to do

**The science.** Say where each planet's two levels sit, in metres, and how far apart they are.
Explain Earth's two levels from the two kinds of crust it is made of, and say what is settled
and what is still argued about Mars's.

**The skills.** A grid of numbers is a **numpy array**: `.shape` to see how big it is,
`.ravel()` to lay it out in one line, `earth < 0` to ask one question of every cell at once,
`np.histogram` to count what falls where, and `plt.imshow` to draw the whole grid as a picture.
A table with named columns is a **pandas DataFrame**: `.info()`, `.head()`, `.isna()`,
`.value_counts()`, `.sort_values()` and `.groupby()`.

**Ten places where you write something: five in class, five at home.** Each one is headed
*Your turn*, with an empty cell under it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

CACHE = "https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data"

def load(start, end, minmag):
    """Fetch one window of the USGS earthquake catalogue; fall back to the cached copy."""
    try:
        return pd.read_csv(f"https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv&orderby=time-asc"
                       f"&starttime={start}&endtime={end}&minmagnitude={minmag}")
    except Exception as e:
        print("live source unreachable, using the cached copy:", type(e).__name__)
        return pd.read_csv(CACHE + "/" + f"week03_{start}_{end}_M{minmag}.csv")

def elevation(planet):
    """Read one planet's 1-degree elevation grid: row 0 is the north, column 0 is -180 degrees."""
    return pd.read_csv(CACHE + "/" + planet + "_elevation.csv", header=None).values


# These three files live in this repository, so CACHE is their home rather than their fallback:
# there is no live server to try first. The catalogue below is the live read.
earth = elevation("earth")
mars = elevation("mars")
coast = pd.read_csv(CACHE + "/coastlines.csv")

quakes = load("2000-01-01", "2026-01-01", 5.5)
bins = np.arange(-10000, 21000, 250)       # 250-metre bins, the same ones for both planets
print("elevation grids:", earth.shape, mars.shape, " catalogue:", quakes.shape)

## A grid, not a list

Every elevation on this planet, one number per one-degree square, is 64,800 numbers. (Earth's
come from NOAA's ETOPO global relief model, Mars's from the MOLA laser altimeter that flew on
Mars Global Surveyor; both were averaged down to one degree so that the files are small enough to
hand round.) A Python list can hold them. But watch what a list does when you ask it for
arithmetic.

In [ ]:
heights_list = [0, 1000, 2000]
heights_array = np.array([0, 1000, 2000])

print("list  times 2:", heights_list * 2)
print("array times 2:", heights_array * 2)

The list did not double anything — it made a longer list, with the same three numbers twice.
That is what `*` means for a list. What you meant is what an **array** does. An array is
a grid of numbers where every cell is the same kind of thing, so one line of arithmetic changes
all of them at once.

That is why the elevation data is an array. And because an array can be two-dimensional, it
keeps the *shape* of the planet: one row per line of latitude, one column per line of longitude.
Row 0 of both files is the northernmost band and column 0 is the westernmost, so the grid is
already the right way up — nothing needs turning over.

In [ ]:
print("shape:", earth.shape)
print("cells:", earth.size)
print("lowest cell: ", earth.min(), "m")
print("highest cell:", earth.max(), "m")

### ✏️ Your turn 1

The Mars grid is already loaded as `mars`. Print its shape, and print its highest cell in
**kilometres** rather than metres — one line of array arithmetic, then `.max()`.

**Use these names**, because the self-check looks for them: `mars_km` for the grid in kilometres.

In [ ]:
# ← your answer here


assert mars_km.shape == mars.shape, "mars_km should be the whole grid, not one number"
print("✓ Mars in kilometres — the highest cell is",
      round(mars_km.max(), 3), "km above the zero level")

## The shape of a planet's surface

A histogram wants one long line of numbers, not a grid. `.ravel()` reads the grid row by row and
lays it out flat — the same 64,800 numbers, in one line instead of 180 of them.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
earth = elevation("earth")
mars = elevation("mars")

In [ ]:
flat = earth.ravel()
print(flat.shape)

Before the histogram, one number. Comparing an array with a number asks the same question of
every cell at once and hands back a grid of True and False. That is a **mask**, and adding one up
counts the Trues, because Python counts True as 1.

### Predict before you run

What fraction of Earth's solid surface lies below sea level? Commit to a number before you run
the next cell — change `my_guess` to whatever you think, then run it.

In [ ]:
my_guess = 0.70

below = earth < 0
fraction_below = below.sum() / earth.size

print("you guessed:", my_guess)
print("this grid says:", f"{fraction_below:.3f}")

0.660 of the cells. Write that down, and be suspicious of it: what you measured is
a fraction of *cells*, and whether a fraction of cells is the same as a fraction of the planet is
a real question. The second part of the homework settles it, and the answer moves.

Now the whole distribution rather than one number. We fix the bins at 250 metres wide and use the
same ones for both planets, so that a peak position means the same thing every time we quote one.
Left to itself `plt.hist` picks its own bins, and the answer you read off moves when it does —
which is the first part of the homework.

In [ ]:
plt.hist(flat, bins=bins)
plt.xlim(-9500, 7000)
plt.xlabel("elevation (m)")
plt.ylabel("number of 1-degree cells")
plt.title("Earth, 64,800 cells, 250 m bins")
plt.show()

Two humps, and a thinly populated gap between them: a broad one a few kilometres down, and a
taller, narrower one sitting right at zero. (Only 423 of the 64,800 cells are exactly 0 m, so
that spike is real low ground, not the grid rounding anything to sea level.) Reading their
positions off the picture by eye is guesswork, so count instead: `np.histogram` does the same counting `plt.hist` does but
hands you the numbers. `counts[i]` is how many cells fell in bin `i`, and `edges` holds the bin
boundaries, so the middle of bin `i` is halfway between `edges[i]` and `edges[i+1]`.

`.argmax()` gives the *position* of the largest value — the same move as week one's
`list.index(max(list))`, in one word.

In [ ]:
counts, edges = np.histogram(flat, bins=bins)
centres = (edges[:-1] + edges[1:]) / 2

print("tallest bin is centred at", centres[counts.argmax()], "m")
print("and holds", counts.max(), "cells")

### ✏️ Your turn 2

That found the taller hump. The other one needs the same three lines applied to a *slice* of the
bins, so write it once as a function and use it twice.

Write `peak_position(grid, lowest, highest)`: it should histogram `grid` on `bins`, keep only the
bins whose centre lies between `lowest` and `highest`, and return the centre of the tallest one
that is left. Give it a docstring. Then print Earth's deep peak (search between -10000 and -1000)
and Earth's shallow peak (search between -1000 and 21000).

**Use these names**, because the self-check looks for them: `peak_position`, `earth_deep`,
`earth_high`.

In [ ]:
# ← your answer here


assert earth_deep < earth_high, "the deep peak should come out below the shallow one"
print("✓ Earth's two levels —", earth_deep, "m and", earth_high, "m,",
      earth_high - earth_deep, "m apart")

## The same numbers, drawn as a map

The mask you built two cells ago has one `True` or `False` per one-degree square — which is to
say, it is already a map. `plt.imshow` draws any grid as a picture, one pixel per cell, and
`extent` tells it what the corners mean in degrees. The coastline goes on top from
`data/coastlines.csv`, exactly as in week one, so you can see whether the mask agrees with it.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
earth = elevation("earth")
below = earth < 0
# Re-run your own Your turn 2 cell as well, the one that defines peak_position and
# sets earth_deep and earth_high. That code is yours, so this cell cannot rebuild it
# for you.

In [ ]:
plt.imshow(below, extent=[-180, 180, -90, 90], cmap="Greys")
plt.plot(coast.lon, coast.lat, color="firebrick", lw=0.6)
plt.xlabel("degrees east")
plt.ylabel("degrees north")
plt.title("Earth: the 42,765 cells below sea level, of 64,800")
plt.show()

The dark region is not a shape anyone had to draw: it is one comparison, `earth < 0`, applied to
every cell. It comes out as the oceans, and the coastline lands on its edge.

So the deep hump in the histogram, centred near -4375 m, is the ocean floor — the
abyssal plains — and the shallow hump near 125 m is low-lying land. The reason
those are two levels rather than one is that Earth is made of two kinds of crust. Ocean crust is
basalt, thin (a few kilometres) and dense; continental crust is granitic, far thicker (tens of
kilometres) and less dense. Both float on the mantle, and a thick light raft floats higher than a
thin heavy one, so the two kinds settle at two different heights. Plate tectonics keeps the
arrangement going: ocean crust is manufactured at mid-ocean ridges and destroyed at subduction
zones within a couple of hundred million years, while the light continental crust is too buoyant
to sink and stays. Water then fills the low level, which is why the boundary between the two
humps sits so close to sea level.

Note what that map is *not* good at. Every cell is drawn the same size, but a one-degree square
at 60° north is only half as wide, east to west, as one at the equator — `cos(60°) = 0.5` — and
at the poles the width goes to nothing. Antarctica along the bottom of the map is stretched
across far more pixels than it deserves. Hold that thought too.

## The other planet

The Mars grid is the same shape as Earth's — 180 by 360, one cell per one-degree square — and it
was measured by a laser altimeter in orbit, which is why it exists at all. Zero on Mars is not a
sea level; there is no sea. It is a reference surface chosen by geodesists, so instead of a
below-and-above mask we colour the whole range. The colour scale is the one normally used for
topography, so the blue end means nothing more than *low* — there is no water on this map.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
mars = elevation("mars")
flat = elevation("earth").ravel()
# Re-run your own Your turn 2 cell as well, the one that defines peak_position and
# sets earth_deep and earth_high. That code is yours, so this cell cannot rebuild it
# for you.

In [ ]:
# the colour scale stops at 4000 m, or the step between the two halves is invisible
plt.imshow(mars, extent=[-180, 180, -90, 90], cmap="terrain", vmin=-4000, vmax=4000)
plt.colorbar(label="elevation (m)")
plt.xlabel("degrees east")
plt.ylabel("degrees north")
plt.title("Mars, 64,800 cells")
plt.show()

There is nothing subtle about that map. The north of the planet is low and smooth, the south is
high and rough, and the step between them runs most of the way round the planet. Planetary scientists call it the **crustal dichotomy**, and the crust under the northern
lowlands is measurably thinner than the crust under the southern highlands.

What Mars does not have is an ocean to fill the low half, or plates to make and destroy crust.
Whatever put that step there did it long ago and left it, and the surface has kept it ever since.

So: two levels here as well. The next cell puts both planets on the same axis, using the same
250-metre bins, so the shapes can be compared rather than described.

In [ ]:
plt.hist(flat, bins=bins, label="Earth")
plt.hist(mars.ravel(), bins=bins, label="Mars", alpha=0.6)   # see-through, or Mars hides Earth
plt.xlim(-9500, 7000)
plt.xlabel("elevation (m)")
plt.ylabel("number of 1-degree cells")
plt.title("Earth and Mars, 64,800 cells each, 250 m bins")
plt.legend()
plt.show()

### ✏️ Your turn 3

Mars runs off the right of that plot: its highest cell is the 20.708 km you printed in
your turn 1, and the axis stops at 7000 m so that the two distributions are both readable.

Use your `peak_position` function on `mars` — the same two searches you ran for Earth — and then
print how far apart each planet's two levels are.

**Use these names**, because the self-check looks for them: `mars_deep`, `mars_high`.

In [ ]:
# ← your answer here


assert mars_deep < mars_high, "the deep peak should come out below the shallow one"
print("✓ Mars's two levels —", mars_deep, "m and", mars_high, "m, a step",
      (mars_high - mars_deep) - (earth_high - earth_deep), "m bigger than Earth's")

### ✏️ Your turn 4

Same reason? Two or three sentences. Use the four peak positions you measured and the two maps —
what the Earth mask looked like, what the Mars map looked like — and say whether one explanation
covers both planets. If it does not, say what Earth has that Mars does not.

*(Double-click this cell and replace this line with your answer.)*

## Where the Mars argument stands

Earth's two levels have a settled explanation, the one the map and the histogram gave you: two
kinds of crust, floating at two heights, made and destroyed by plate tectonics. Mars's do not.
Which process put that hemisphere-wide step in the crust is genuinely unsettled, and the two
families of explanation under discussion are a single enormous impact that excavated the northern
lowlands, and a pattern of convection inside the young planet that thinned the crust on one side.
Nobody has closed the argument, so if you found the Mars half less satisfying than the Earth half,
that is not because the notebook left something out.

## When the data has names

An array is the right container when every number means the same thing — here, metres of
elevation — and position is what tells them apart. Plenty of data is not like that. An earthquake
catalogue has a time, a latitude, a depth, a magnitude and a place name on every row, and those
are five different kinds of thing. What that wants is a **table**: a table with a name on every
column, so you ask for data by name instead of by position. Pandas calls one a DataFrame.

The catalogue below is every earthquake of magnitude 5.5 and above that the USGS has recorded
since the start of 2000. Remember what such a file is: *A catalogue lists what somebody's
instruments recorded, not what happened. Where there are no seismometers there are no earthquakes
in the file.*

`.info()` is the first thing to run on a table you have not seen. It names every column, says
what type it holds, and — the part that matters — says how many rows are not blank.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
quakes = load("2000-01-01", "2026-01-01", 5.5)

In [ ]:
print(quakes.shape)
quakes.info()

print("rows with no dmin:", quakes["dmin"].isna().sum())
print("rows left if you drop every row with any blank:", len(quakes.dropna()))

12,849 rows and 22 columns. Read down the non-null counts: `time`,
`latitude`, `depth` and `mag` are complete, but several columns are not. Where the file had
nothing at all, pandas puts NaN. A NaN is a hole, not a zero. The difference matters: a depth of
0 km is a real, shallow earthquake, while a NaN depth is an earthquake whose depth nobody
recorded.

`.isna()` marks the holes — it is exactly the mask move from the first half of the notebook, with
`is this blank?` as the question instead of `is this below zero?`. `.dropna()` throws away every
row with a hole anywhere in it, which sounds tidy and is usually a disaster: only
1,673 rows survive out of 12,849, not because those earthquakes are
bad but because a column nothing here needs happens to be patchy. So keep the columns you
actually want, and leave the rest alone.

A new column is made by assigning to a name that does not exist yet. `.str[:4]` slices every
string in a column the way `[:4]` slices one string, so the four characters at the front of
`time` give the year. Then the columns worth keeping are chosen by name, in a list, inside the
square brackets, and `.head()` shows the first five rows.

In [ ]:
quakes["year"] = quakes["time"].str[:4]
quakes = quakes[["year", "depth", "mag", "type", "place"]]

print(quakes.head())

`.value_counts()` counts how often each value appears in one column — the table version of
counting `True`s in a mask. Run it on `type` and the catalogue tells you something about itself.

In [ ]:
print(quakes["type"].value_counts())
print(quakes[quakes["type"] != "earthquake"])

Not everything in the earthquake catalogue is an earthquake. 2 of the
12,849 rows are not: seismometers record whatever shakes the ground, and a large enough
explosion or eruption shakes it in much the way an earthquake does. Telling those apart is a
question this course comes back to.

That second line is **boolean filtering**: `quakes["type"] != "earthquake"` is a mask, one True or
False per row, and putting a mask inside the square brackets keeps the rows where it is True. The
grid and the table use the same move. `.sort_values()` then puts the rows in order by a column.

In [ ]:
big = quakes[quakes["mag"] >= 8.0]
print(len(big), "earthquakes at magnitude 8.0 or above")
print(big.sort_values("mag", ascending=False).head()[["year", "mag", "place"]])

### ✏️ Your turn 5

Earthquakes deeper than 500 km are strange animals: the rock at that depth is far too hot and
squeezed to snap the way it does near the surface, and they only happen where cold ocean floor is
sinking back into the mantle — the far end of the story the first half of this notebook told.

Filter `quakes` to the rows deeper than 500 km, print how many there are, and print the five
deepest in order, deepest first, showing the year, the depth and the place.

**Use these names**, because the self-check looks for them: `deep_quakes`.

In [ ]:
# ← your answer here


assert deep_quakes["depth"].min() > 500, "deep_quakes should hold only the rows below 500 km"
print("✓ deep earthquakes —", len(deep_quakes), "of them, the deepest at",
      deep_quakes["depth"].max(), "km")

## The question, answered

**No — two levels, two different causes.** Earth's two levels are two kinds of crust, thin dense
ocean floor and thick light continent, floating at two heights and continuously remade by plate
tectonics; Mars's are one hemisphere-wide step in crustal thickness, made once and never reworked,
on a planet with neither ocean nor plates. The table half is the same story from its other end:
the earthquakes you filtered out below 500 km happen where ocean floor is sinking back into the
mantle, which is the half of the cycle that keeps Earth's low level low.

## Week 3 summary

**The question.** Earth's elevation has two peaks. So does Mars's. Same reason?

### What to remember

| | |
|---|---|
| **1** | Earth's surface has two levels — ocean floor and continent. Mars has two as well, for a different reason nobody has settled. |
| **2** | A histogram's bin count can hide the science: too few bins merge real peaks. |
| **3** | A longitude-latitude grid over-counts the poles, so area-weight before quoting any percentage. |

### The ideas, in plain words

| Idea | Means |
|---|---|
| **Array** | A grid of numbers where every cell is the same kind of thing, so one line of arithmetic changes all of them at once. |
| **Mask** | Comparing an array with a number asks the same question of every cell at once and hands back a grid of True and False. |
| **NaN** | Where the file had nothing at all, pandas puts NaN. A NaN is a hole, not a zero. |
| **Table** | A table with a name on every column, so you ask for data by name instead of by position. |
| **Area weighting** | A longitude-latitude grid counts every square once, but a square near the pole is a sliver; weight each row by cos(latitude) before quoting a percentage. |

### Code you met this week

| Function | What it does |
|---|---|
| `np.array(list)` | the container that does arithmetic to every number at once |
| `grid.shape` | how many rows and columns |
| `grid.size` | how many numbers altogether |
| `grid.ravel()` | lay a grid out flat, as one long line of numbers |
| `grid.min() / grid.max()` | smallest / largest value in the whole grid |
| `grid.sum()` | add every number up — on a mask, that counts the Trues |
| `grid.argmax()` | the position of the largest value, not the value itself |
| `np.arange(start, stop, step)` | evenly spaced numbers — here, the bin edges |
| `np.histogram(values, bins=edges)` | the counts a histogram would draw, handed back as numbers |
| `plt.imshow(grid, extent=[...])` | draw a whole grid as a picture, one pixel per cell |
| `plt.colorbar(label=...)` | the key that says what the colours mean |
| `table.info()` | every column, its type, and how many rows are not blank |
| `table.head()` | the first five rows |
| `column.isna()` | a mask marking where the file had nothing |
| `table.dropna()` | throw away every row with a hole anywhere in it |
| `column.value_counts()` | how often each value appears |
| `table.sort_values(by)` | put the rows in order by one column |
| `table.groupby(column)` | split the table into one group per value |
| `column.count() / column.median()` | how many, and the middle value |
| `table.loc[label]` | read one row out by its label |

## Homework

Three parts, all on the two grids and the catalogue you already have loaded. Part 1 and part 2 go
back to the elevation grids and finish two arguments class deliberately left open; part 3 stays
with the table. If you have restarted since class, run the setup cell at the top and then the
checkpoint below: between them they rebuild everything the three parts read.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
earth = elevation("earth")
mars = elevation("mars")
below = earth < 0
fraction_below = below.sum() / earth.size

quakes = load("2000-01-01", "2026-01-01", 5.5)
quakes["year"] = quakes["time"].str[:4]
quakes = quakes[["year", "depth", "mag", "type", "place"]]

### ✏️ Your turn 6

Class fixed the bins at 250 metres wide. Loosen that, and the two peaks eventually merge into one
hump — but which way? Find out.

Loop over `bin_counts = [3, 4, 5, 6, 8, 10, 20]` and for each one print the bin count and the counts
`np.histogram(earth.ravel(), bins=n)` returns. You are looking for a **dip** between two larger
numbers: that dip is the gap between the two levels, and when it disappears the peaks have
merged. Report the smallest number of bins that still shows two humps for Earth, and then do the
same for Mars.

**Use these names**, because the self-check looks for them: `earth_fewest_bins`,
`mars_fewest_bins`.

In [ ]:
# ← your answer here


assert earth_fewest_bins in bin_counts, "pick one of the bin counts you actually tried"
assert mars_fewest_bins in bin_counts, "pick one of the bin counts you actually tried"
print("✓ where the peaks merge — Earth keeps two humps down to", earth_fewest_bins,
      "bins, Mars down to", mars_fewest_bins)

### ✏️ Your turn 7

Class measured that 0.660 of the *cells* in the Earth grid are below sea level,
and parked the question of whether that is the fraction of the *planet*. Here is the missing
piece, and it has a name — **area weighting**. A longitude-latitude grid counts every square
once, but a square near the pole is a sliver; weight each row by cos(latitude) before quoting a
percentage. A square at latitude *lat* is `cos(lat)` times as wide, east to west, as one at the
equator.

So count each row of the grid by how wide its cells are instead of by how many there are:

```
lats = np.arange(89.5, -90, -1)          # the centre latitude of each of the 180 rows
cell_width = np.cos(np.deg2rad(lats))    # 1.0 at the equator, nearly 0 at the poles
rows_below = below.sum(axis=1)           # below-sea-level cells in each row
```

`rows_below` and `cell_width` are both 180 numbers long, so `rows_below * cell_width` weights
each row. Divide by what all 180 full rows would weigh — `360 * cell_width.sum()` — to get the
weighted fraction. Then say, in the cell after, which of the two numbers is the honest one and
what the longitude-latitude grid was doing to the poles to produce the other.

**Use these names**, because the self-check looks for them: `earth_weighted`.

In [ ]:
# ← your answer here


assert earth_weighted != fraction_below, "if nothing moved, the weights were not used"
print("✓ area weighting — the fraction below sea level moved from",
      f"{fraction_below:.3f}", "to", f"{earth_weighted:.3f}")

*(Double-click this cell and replace this line with your answer.)*

### ✏️ Your turn 8

Back to the table. `quakes` has a `year` column, so `.groupby("year")` will split the catalogue
into one group per year, and `["mag"].count()` will count the rows in each group.

Build `per_year`, print the three largest years using `.sort_values(ascending=False).head(3)`,
and then print the year before and the year after the busiest one — `per_year.loc["2007"]` reads
one year out — together with `per_year.median()`.

Then, in the cell after, answer this in two or three sentences using your own numbers: was the
planet busier in that year, or did one thing happen? Whatever you claim, the neighbouring years
and the median have to be consistent with it.

**Use these names**, because the self-check looks for them: `per_year`.

In [ ]:
# ← your answer here


assert len(per_year) == 26, "one row per year is expected, from 2000 to 2025"
print("✓ earthquakes by year — busiest is",
      per_year.sort_values(ascending=False).index[0], "with", per_year.max(),
      "against a median of", per_year.median())

*(Double-click this cell and replace this line with your answer.)*